In [ ]:
# Gearbox-style measurement errors on the actor's joint encoders (api/blind_repose.py MeasurementModel; schemes from Origami Robotics' "The
# Dexterity Deadlock": backlash, stiction dead zone ("information wall"), and a time-varying "chaos" term), scaled with the gear ratio N
# (dead-band widths are our assumption: h = d_s = 0.1 deg * N/15 at K = 1 N m/rad; chaos = per-joint per-episode log-normal width factor
# plus a slow drift). A CLEAN trained policy is evaluated at inference time under each condition (no retraining) with the usual full-episode
# protocol (greedy, 4096 episodes after discarding each env's first). Env vars: PPO_VARIANT (proprio | touch | privileged), PPO_CKPT_PATH
# (relative to control/), SCHEMES (comma list of clean, backlash, stiction, bs = backlash + stiction, both = bs + chaos), NS (gear ratios), GPU.
import os

os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("GPU", "0")
from api import isaac

app = isaac.launch(headless=True)

In [ ]:
import hashlib
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

import api.blind_repose  # noqa: F401  (registers the blind tasks)
from api import policies
from api.joint_torque import enable_contact_reporting
from api.repose_eval import evaluate_policy
from utils.experiment_cache import ExperimentCache
from utils.path import ROOT_DIR

VARIANT = os.environ.get("PPO_VARIANT", "proprio")
CKPT = os.environ.get("PPO_CKPT_PATH", "logs/rl_games/blind_repose/blind-proprio/nn/last_blind_repose_ep_5000_rew_3925.8337.pth")
SCHEMES = os.environ.get("SCHEMES", "clean,backlash,stiction,both").split(",")
NS = [float(x) for x in os.environ.get("NS", "15,50,100,200,288,600,1200").split(",")]
TASK = {"proprio": "Isaac-Repose-Cube-Shadow-Blind-Proprio-v0", "touch": "Isaac-Repose-Cube-Shadow-Blind-Touch-v0",
        "privileged": "Isaac-Repose-Cube-Shadow-Privileged-v0"}[VARIANT]
NUM_ENVS, EVAL_EPISODES, EVAL_SEED = int(os.environ.get("MEAS_NUM_ENVS", "2048")), 2, 1
ckpt_path = ROOT_DIR / CKPT
assert ckpt_path.exists(), ckpt_path
env = isaac.make_env(TASK, num_envs=NUM_ENVS, cfg_fn=enable_contact_reporting if VARIANT == "touch" else None)
u = env.unwrapped
device = u.device
policy = policies.load_rl_games(TASK, checkpoint=ckpt_path, device=device)
digest = hashlib.sha256(ckpt_path.read_bytes()).hexdigest()[:16]
print(f"{TASK}: {NUM_ENVS} envs, checkpoint {CKPT} ({digest}), schemes {SCHEMES}, N in {NS}")

In [ ]:
# how big are the dead zones relative to the deflections the policy sees? Clean rollout; |q - target| over the actuated joints is the PD
# deflection (tracking lag during motion plus contact). Fractions are for finger joints (K = 1 N m/rad; the wrist, K = 5, has 5x narrower dead zones).
import math

fing = torch.tensor([("WRJ" not in u.hand.joint_names[i]) for i in u._meas.act.tolist()], device=device)
u.set_measurement(n=0)
obs, _ = env.reset(seed=EVAL_SEED)
defl = []
with torch.no_grad():
    for _ in range(300):
        obs, *_ = env.step(policy(obs))
        defl.append((u.hand_dof_pos[:, u._meas.act] - u.cur_targets[:, u._meas.act]).abs())
d = torch.cat(defl)
print("clean |q - target| (deg), actuated joints: median %.1f, p75 %.1f, p90 %.1f, p99 %.1f" % tuple(
    torch.rad2deg(d.flatten().quantile(torch.tensor([0.5, 0.75, 0.9, 0.99], device=device))).tolist()))
for N in [15, 50, 100, 200, 288, 600, 1200]:
    ds = math.radians(0.1) * N / 15
    print(f"  N={N:5.0f}: backlash half-width / stiction dead zone {math.degrees(ds):5.2f} deg (finger), hides {100 * (d[:, fing] < ds).float().mean():5.1f}% of finger deflections")

In [ ]:
def evaluate(cfg, seed=EVAL_SEED):
    """Full-episode eval (api/repose_eval.py) of the policy under one measurement-error condition."""
    u.set_measurement(**cfg)
    try:
        return evaluate_policy(env, policy, episodes=EVAL_EPISODES, seed=seed)
    finally:
        u.set_measurement(n=0)


CONDS = {"clean": dict(n=0), "backlash": lambda N: dict(n=N, backlash=True), "stiction": lambda N: dict(n=N, stiction=True),
         "bs": lambda N: dict(n=N, backlash=True, stiction=True),  # backlash + stiction, no chaos
         "both": lambda N: dict(n=N, backlash=True, stiction=True, chaos=True)}
cache = ExperimentCache()
# the cache key below holds the measurement CONFIG but not the MeasurementModel constants: record them once and refuse to mix results
# from different constants (if they change on purpose, bump `version` in cond_params)
import json

_MC = {k: getattr(api.blind_repose.MeasurementModel, k) for k in ("N_REF", "H0_DEG", "TAU0", "CHAOS_SIGMA", "DRIFT_SIGMA", "DRIFT_TAU")}
_mc_path = cache.dir / "meas_model_constants.json"
if _mc_path.exists():
    assert json.loads(_mc_path.read_text()) == _MC, f"MeasurementModel constants changed: {_MC}; bump `version` in cond_params (cached results ignore them)"
else:
    _mc_path.write_text(json.dumps(_MC))


def cond_params(scheme, N):
    cfg = CONDS[scheme] if scheme == "clean" else CONDS[scheme](N)
    return cfg, dict(version=1, task=TASK, ckpt=CKPT, digest=digest, num_envs=NUM_ENVS, episodes=EVAL_EPISODES, seed=EVAL_SEED, cfg=cfg)


se = lambda a: a.std(ddof=1) / np.sqrt(len(a))
for scheme in SCHEMES:
    for N in ([0] if scheme == "clean" else NS):
        cfg, params = cond_params(scheme, N)

        def run_eval(cfg=cfg):
            t0 = time.time()
            r, g, l = evaluate(cfg)
            print(f"  ({len(r)} episodes in {time.time() - t0:.0f}s)", flush=True)
            return {"return": r, "goals": g, "length": l}

        ev = cache.compute("repose_blind_meas_sweep", params, run_eval)
        r, g, l = ev["return"], ev["goals"], ev["length"]
        print(f"SWEEP {VARIANT:10s} {scheme:9s} N={N:6.0f}  return {r.mean():8.1f} +- {se(r):5.1f}  goals/ep {g.mean():5.2f} +- {se(g):4.2f}  "
              f"ended early {100 * (l < u.max_episode_length - 1).mean():5.1f}%", flush=True)

In [ ]:
# summary from the cache (all schemes evaluated so far for this checkpoint, also by other processes): return and goals against the gear ratio
res = {}
for scheme in ["clean", "backlash", "stiction", "bs", "both"]:
    for N in ([0] if scheme == "clean" else NS):
        _, params = cond_params(scheme, N)
        ev = cache.get(cache.make_key("repose_blind_meas_sweep", params))
        if ev is not None:
            res[(scheme, N)] = (ev["return"].mean(), se(ev["return"]), ev["goals"].mean(), se(ev["goals"]))
clean = res.get(("clean", 0))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for scheme, label in [("backlash", "backlash"), ("stiction", "stiction dead zone"), ("bs", "backlash + stiction"), ("both", "backlash + stiction + chaos")]:
    pts = [(N, *res[(scheme, N)]) for N in NS if (scheme, N) in res]
    if pts:
        x = [p[0] for p in pts]
        ax[0].errorbar(x, [p[1] for p in pts], [p[2] for p in pts], marker="o", label=label)
        ax[1].errorbar(x, [p[3] for p in pts], [p[4] for p in pts], marker="o", label=label)
if clean:
    for a, v in zip(ax, [clean[0], clean[2]]):
        a.axhline(v, color="k", ls="--", label="clean")
for a, t in zip(ax, ["episode return", "goals reached / episode"]):
    a.set(xlabel="gear ratio N (dead zones scale with N)", ylabel=t, xscale="log", title=f"{VARIANT}: {t}")
    a.grid(alpha=0.3)
    a.legend()
plt.tight_layout()
plt.show()